# Build the GISIdeas Asia conference deck — "GISIdeas_Asia_AlphaEarth_IMD.pptx"

Packaged from three scripts. The first (`build_gisideas_deck.py`) fully
**regenerates** the deck from the Politecnico template — running it discards
any manual edits made in PowerPoint since the last generation. The other two
patch the live file in place (append one slide, preserving hand-edits) and
should only be run after any manual edits are done, in this order.

Requires `figs_deck/all_predictions_grid.png` (see
`build_all_predictions_grid.ipynb`) and the six `report/figs/*.png` /
`figs_deck/*.png` result figures already on disk.

Close the .pptx in PowerPoint (and let OneDrive finish syncing) before
running any cell here.

## 1. Build the 22-slide deck from the Polimi template (full regeneration)

**Regenerates from scratch — do not run this after manual PowerPoint edits without expecting to lose them.**

In [ ]:
# -*- coding: utf-8 -*-
"""Build the GISIdeas Asia conference deck on the Politecnico (violet) template.

Requested by Prof. Brovelli's email of 2026-09-11: ~20-25 min / ~20 slides,
title "Mapping Imperviousness Density Using Geospatial Foundation-Model
AlphaEarth Embeddings", authors Brovelli / Žgela / Kirubakaran / Tan.

Every slide is built fresh from a named layout in the template (never a
copy-edit of the template's own Lorem-ipsum example slides), so the deck
carries only this talk's content while keeping the template's fonts, colours
and placeholder geometry. Numbers are the ones already established in this
project's report and notebook 05 cross-city CSVs; figures are the existing
report/figs/*.png and figs_deck/*.png (no notebook re-run).

    C:\\ProgramData\\anaconda3\\python.exe build_gisideas_deck.py
"""
import sys
sys.stdout.reconfigure(encoding="utf-8")

from PIL import Image
from pptx import Presentation
from pptx.util import Emu, Inches

TEMPLATE = (r"C:\Users\user\OneDrive - Politecnico di Milano\Keerthana_phd_folder"
            r"\Annual report\Template_ppt_2024_DICA_v1.pptx")
OUT = (r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious"
       r"\Ground_truth_validation_S2\Presentation\GISIdeas_Asia_AlphaEarth_IMD.pptx")

REPORT_FIGS = "report/figs"
DECK_FIGS = "figs_deck"

DATE_TXT = "GISIdeas Asia  \u00b7  Hanoi, Vietnam"
FOOTER_TXT = "Mapping IMD with AlphaEarth Embeddings"

prs = Presentation(TEMPLATE)
M0, M1 = prs.slide_masters[0], prs.slide_masters[1]


def layout(master, name):
    for lay in master.slide_layouts:
        if lay.name == name:
            return lay
    raise KeyError(name)


L_TITLE = layout(M1, "Cover G")
L_AGENDA = layout(M0, "Indice_A")
L_DIVIDER = layout(M0, "Divisorio_A")
L_TEXT = layout(M0, "Pagina base_ testo A")
L_TEXT2 = layout(M0, "Pagina base_ testo B")
L_TABLE = layout(M0, "Pagina base_Oggetto")
L_TABLE_TEXT = layout(M0, "Pagina colonna _ oggetto")
L_IMAGE = layout(M0, "Pagina immagine _ A")
L_EVIDENZA = layout(M0, "Pagina Evidenza _ A")
L_FINALE = layout(M0, "Slide_finale_B")
L_CONTACT = layout(M1, "Cover Finale")

# Drop every example slide the template ships with -- we build only new ones.
# Removing just the sldIdLst entry leaves the relationship (and so the part)
# reachable, which collides with the part names add_slide() hands out next;
# drop the relationship too so the old parts are actually pruned on save.
xml_slides = prs.slides._sldIdLst
for sld_id in list(xml_slides):
    prs.part.drop_rel(sld_id.get(
        "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"))
    xml_slides.remove(sld_id)


def add(lay):
    return prs.slides.add_slide(lay)


def meta(slide, date=DATE_TXT, footer=FOOTER_TXT):
    for idx, val in ((10, date), (11, footer)):
        if idx in [p.placeholder_format.idx for p in slide.placeholders]:
            slide.placeholders[idx].text = val


def settxt(slide, idx, text):
    slide.placeholders[idx].text = text


def fill_body(placeholder, lines):
    tf = placeholder.text_frame
    tf.word_wrap = True
    for i, line in enumerate(lines):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.text = line


def add_table(slide, idx, headers, rows):
    ph = slide.placeholders[idx]
    x, y, w, h = ph.left, ph.top, ph.width, ph.height
    ph._element.getparent().remove(ph._element)
    gframe = slide.shapes.add_table(len(rows) + 1, len(headers), x, y, w, h)
    table = gframe.table
    for j, hd in enumerate(headers):
        table.cell(0, j).text = str(hd)
    for i, row in enumerate(rows):
        for j, v in enumerate(row):
            table.cell(i + 1, j).text = str(v)
    return table


def add_picture_fit(slide, path, x, y, w, h):
    iw, ih = Image.open(path).size
    ar = iw / ih
    box_ar = w / h
    if ar > box_ar:
        pw, ph = w, Emu(int(w / ar))
    else:
        ph, pw = h, Emu(int(h * ar))
    px = x + Emu(int((w - pw) / 2))
    py = y + Emu(int((h - ph) / 2))
    slide.shapes.add_picture(path, px, py, width=pw, height=ph)


def image_slide(title, img_path, caption_lines, date=DATE_TXT, footer=FOOTER_TXT):
    s = add(L_IMAGE)
    settxt(s, 0, title)
    meta(s, date, footer)
    fill_body(s.placeholders[14], caption_lines)
    add_picture_fit(s, img_path, Inches(6.15), Inches(0.95), Inches(6.85), Inches(5.95))
    return s


def big_image_slide(title, img_path, date=DATE_TXT, footer=FOOTER_TXT):
    """Title strip + one large centred image spanning nearly the full slide
    width, for a figure too busy for the text-beside-image layout."""
    s = add(L_TEXT)
    settxt(s, 0, title)
    meta(s, date, footer)
    body = s.placeholders[13]
    body._element.getparent().remove(body._element)
    add_picture_fit(s, img_path, Inches(0.37), Inches(2.02), Inches(12.60), Inches(5.15))
    return s


def text_slide(title, lines, date=DATE_TXT, footer=FOOTER_TXT):
    s = add(L_TEXT)
    settxt(s, 0, title)
    meta(s, date, footer)
    fill_body(s.placeholders[13], lines)
    return s


def text2_slide(title, left_head, left_lines, right_head, right_lines,
                 date=DATE_TXT, footer=FOOTER_TXT):
    s = add(L_TEXT2)
    settxt(s, 0, title)
    meta(s, date, footer)
    fill_body(s.placeholders[13], [left_head, ""] + left_lines)
    fill_body(s.placeholders[14], [right_head, ""] + right_lines)
    return s


def divider(title, number, date=DATE_TXT, footer=FOOTER_TXT):
    s = add(L_DIVIDER)
    settxt(s, 0, title)
    meta(s, date, footer)
    settxt(s, 13, number)
    return s


# ===================================================================== 1. title
s = add(L_TITLE)
settxt(s, 22, "GISIdeas Asia  \u00b7  Hanoi, Vietnam")
settxt(s, 0, "Mapping Imperviousness Density Using Geospatial\n"
             "Foundation-Model AlphaEarth Embeddings")
settxt(s, 21, "Maria Antonia Brovelli, Matej \u017d" + "gela, Keerthana Kirubakaran, Xiao Tao  "
              "\u00b7  Politecnico di Milano")

# ===================================================================== 2. agenda
s = add(L_AGENDA)
settxt(s, 0, "Outline")
meta(s)
fill_body(s.placeholders[13], ["Milan \u00b7 Hanoi \u00b7 Ho Chi Minh City", "2018 \u00b7 10 m"])
fill_body(s.placeholders[15], [
    "1.  Why map impervious surfaces",
    "2.  Data and method: AlphaEarth vs Sentinel-2",
    "3.  Milan: same-source validation",
    "4.  Vietnam: does it transfer?",
    "5.  Independent validation: the reality check",
    "6.  Conclusions",
])

# ===================================================================== 3. divider 01
divider("Why map impervious surfaces", "01")

# ===================================================================== 4. motivation
text_slide("Why map impervious surfaces", [
    "Impervious surface density (IMD): the sealed fraction of the ground \u2014 "
    "roofs, roads, pavements.",
    "A key indicator for urban growth, flood risk and the urban heat island.",
    "Global Earth-observation products already map it, but they are built from "
    "hand-designed spectral indices.",
    "This work asks two questions: can a geospatial foundation model's embeddings "
    "map IMD as well as a purpose-built Sentinel-2 composite, and does the answer "
    "travel to cities the model never trained on?",
])

# ===================================================================== 5. data & study areas
text2_slide(
    "Three cities, two reference products",
    "Study areas",
    ["Milan, Hanoi, Ho Chi Minh City.",
     "2018 Sentinel-2 L2A composites, 10 m.",
     "Milan: 30 usable dates. Hanoi: 4. HCMC: 3 \u2014 an archive limit, not a "
     "modelling choice."],
    "Training labels",
    ["Milan: CLMS Imperviousness Density \u2014 sealed surface, Europe only.",
     "Hanoi, HCMC: GHS-BUILT-S \u2014 built-up surface, global, but excludes roads "
     "and pavements by design.",
     "Milan was later modelled against GHS-BUILT-S too, to measure that "
     "definitional gap directly."],
)

# ===================================================================== 6. AlphaEarth vs S2
text2_slide(
    "AlphaEarth embeddings vs Sentinel-2 composites",
    "AlphaEarth embeddings",
    ["A 64-band annual satellite-embedding foundation model (Google DeepMind).",
     "Learned from multi-sensor Earth observation, used here whole \u2014 60 of 64 "
     "bands separate the IMD classes significantly (\u017d" + "gela, 2025).",
     "One feature set, no composite choice to make."],
    "Sentinel-2 composites",
    ["A hand-built alternative, tested at three depths: median (10 bands), stack "
     "(4 seasonal dates, 40 bands), percentile (p10\u2013p90, 50 bands).",
     "Only the median composite is computable in Vietnam \u2014 a percentile needs "
     "at least 17 cloud-free dates and Hanoi and HCMC do not reach it."],
)

# ===================================================================== 7. method
text_slide("Method: sampling, cross-validation, two validations", [
    "Random forest, trained on a stratified spatial sample (3 500 points per "
    "city, 500 in each of 7 IMD classes).",
    "Spatial block cross-validation with a 250 m buffer, so training and test "
    "points never share a neighbourhood (design reused from \u017d" + "gela, 2025).",
    "Every model is checked twice: same-source, against the product it was "
    "trained on; and independently, against 450 photo-interpreted plots per "
    "city that no model ever saw.",
])

# ===================================================================== 8. divider 02
divider("Milan: same-source validation", "02")

# ===================================================================== 9. Milan table
s = add(L_TABLE)
settxt(s, 0, "Against CLMS, all three Sentinel-2 composites beat the embeddings")
meta(s)
add_table(s, 1,
          ["Predictor set", "RMSE", "MAE", "R\u00b2", "Bias"],
          [["S2 percentile (50 bands)", "9.46", "6.39", "0.93", "\u22120.10"],
           ["S2 stack (40 bands)", "10.86", "7.64", "0.90", "\u22120.17"],
           ["S2 median (10 bands)", "11.29", "7.82", "0.90", "0.05"],
           ["AlphaEarth embeddings (64 bands)", "14.12", "10.68", "0.84", "0.62"]])
settxt(s, 16, "Note: random forest, 1 014-point spatial holdout, scored against CLMS. "
              "Percentile composite cuts RMSE by 33% against the embeddings, with the "
              "same ordering on RMSE, MAE and R\u00b2.")

# ===================================================================== 10. Milan maps
image_slide(
    "What the disagreement with CLMS looks like",
    f"{REPORT_FIGS}/fig_milan_raster_comparison.png",
    ["Observed CLMS imperviousness, the S2-percentile prediction, and their "
     "difference.",
     "The two upper maps agree on the shape of the conurbation: the dense core, "
     "the satellite towns, the agricultural south.",
     "The model reads narrow roads and canals as slightly more impervious than "
     "CLMS, and the very centre slightly less \u2014 a resolution disagreement, "
     "not a systematic error."],
)

# ===================================================================== 11. divider 03
divider("Vietnam: does it transfer?", "03")

# ===================================================================== 12. zero-shot vs retrain
s = add(L_TABLE_TEXT)
settxt(s, 0, "Zero-shot fails; local retraining repairs most of it")
meta(s)
fill_body(s.placeholders[13], [
    "A Milan-trained model applied directly to Hanoi and HCMC (zero-shot) fails: "
    "R\u00b2 at or below zero in three of four city/predictor combinations.",
    "Retraining locally on GHS-BUILT-S recovers it. S2 median composites fail "
    "the same way and recover the same way.",
])
add_table(s, 15,
          ["City", "Scenario", "RMSE", "R\u00b2"],
          [["Hanoi", "zero-shot (AlphaEarth)", "35.96", "\u22120.07"],
           ["Hanoi", "local retrain", "23.86", "0.53"],
           ["HCMC", "zero-shot (AlphaEarth)", "40.65", "\u22120.28"],
           ["HCMC", "local retrain", "21.31", "0.65"]])
settxt(s, 16, "Source: same-source validation against GHS-BUILT-S, spatial holdout.")

# ===================================================================== 13. bias recovery
image_slide(
    "Why transfer fails, part 1: the level is wrong",
    f"{REPORT_FIGS}/fig_bias_recovery.png",
    ["Zero-shot models carry Milan's high average level into a lower-level city.",
     "GHS-BUILT-S itself under-marks the independently interpreted reference by "
     "about 20 percentage points, in both cities \u2014 the road exclusion showing "
     "up as a number.",
     "Retraining does not erase that gap, but it recovers 38\u201367% of it: the "
     "Sentinel-2 and embedding features still see the roads the label omits."],
)

# ===================================================================== 14. lost low tail
image_slide(
    "Why transfer fails, part 2: the low tail disappears",
    f"{REPORT_FIGS}/fig_hcmc_prediction_histogram.png",
    ["HCMC's photo-interpreted reference is bimodal: many plots near 0%, many "
     "near 100%.",
     "The zero-shot embeddings map has no low tail at all \u2014 it is empty "
     "below 20% to four decimal places.",
     "Every local retrain restores it. That is a shape failure, not only a "
     "level shift, and no simple correction could have repaired it."],
)

# ===================================================================== 15. divider 04
divider("Independent validation: the reality check", "04")

# ===================================================================== 16. validation design
text_slide("450 plots no model ever saw", [
    "450 photo-interpreted plots per city, 1 350 in total, drawn independently "
    "in Google Earth Pro on 2018 imagery.",
    "Every raster is scored on the same terms, including CLMS and GHS-BUILT-S "
    "themselves \u2014 scored here as maps under test, not as ground truth.",
    "Three complementary techniques: continuous error, a hard pervious/"
    "impervious cut at 50%, and a threshold-free 10-class ordinal agreement. "
    "All three tell the same story.",
])

# ===================================================================== 17. compression (KEY)
image_slide(
    "Change the reference, and the Milan spread compresses 6.4-fold",
    f"{REPORT_FIGS}/fig_samesource_vs_independent.png",
    ["Same predictors, same four Milan models, two references.",
     "Against CLMS the spread is 4.66 RMSE, 33% of the worst map. Against "
     "independent photo-interpretation it is 1.34, or 5%.",
     "Most of what same-source validation measured as AlphaEarth trailing "
     "Sentinel-2 was agreement with the training label, not real accuracy."],
)

# ===================================================================== 18. training label (NEW)
image_slide(
    "Does the training label itself matter? Milan says yes",
    f"{DECK_FIGS}/trainlabel_deck.png",
    ["The same four Milan feature sets, refit on GHS-BUILT-S instead of CLMS, "
     "scored on the same 450 plots.",
     "Every metric drops: Cohen's kappa from about 0.66 to 0.49, QWK from 0.78 "
     "to 0.68, and bias flips from \u22123 to about +9 to +10 percentage points.",
     "The features still see the roads GHS-BUILT-S omits, so the model recovers "
     "about half the label's deficit \u2014 the same mechanism seen in Vietnam, "
     "now shown on the same city and the same features."],
)

# ===================================================================== 19. cross-city summary
image_slide(
    "Twenty rasters, three cities, one reference",
    f"{DECK_FIGS}/rmse.png",
    ["Every raster used in this study, scored against independent "
     "photo-interpretation, grouped by city.",
     "CLMS-trained Milan models lead; every GHS-BUILT-S-trained model, in any "
     "city, sits behind its CLMS-trained counterpart and ahead of the raw "
     "GHS-BUILT-S product.",
     "The same three-tier pattern \u2014 trained models, transferred or "
     "relabelled models, raw benchmark \u2014 holds in every city."],
)

# ================================================================= 19b. all maps
big_image_slide(
    "What all twenty predictions look like",
    f"{DECK_FIGS}/all_predictions_grid.png",
)

# ===================================================================== 20. conclusions
text2_slide(
    "Conclusions",
    "What this shows",
    ["Composite choice matters more than the foundation model, same-source: "
     "percentile composites cut RMSE by 33% against the embeddings.",
     "Independently validated, that spread compresses about 6-fold. Same-source "
     "validation is the right tool for choosing a model and the wrong one for "
     "claiming its accuracy.",
     "Zero-shot transfer fails; local retraining is necessary and mostly, not "
     "fully, repairs it.",
     "Training-label quality bounds what any model can recover \u2014 about "
     "half of GHS-BUILT-S's omission, never all of it."],
    "Limits and next steps",
    ["Composite depth and predictor type are confounded in Vietnam: 30 "
     "Sentinel-2 dates in Milan against 3\u20134 in Hanoi and HCMC.",
     "Plot independence and interpreter effects are assumed, not tested.",
     "Next: an independent Vietnamese reference that includes roads, and a "
     "deeper 2018 Sentinel-2 archive as it is reprocessed."],
)

# ===================================================================== 21. thank you
s = add(L_FINALE)
settxt(s, 0, "Thank you")

# ===================================================================== 22. contact
s = add(L_CONTACT)
settxt(s, 23, "Indirizzo / 02 2399 0000 / mail@polimi.it / www.polimi.it")

prs.save(OUT)
print("saved:", OUT)
print("slides:", len(prs.slides))

## 2. Add the all-predictions map grid (in place, preserves manual edits)

In [ ]:
# -*- coding: utf-8 -*-
"""Add ONE new slide -- the all-twenty-predictions map grid -- to the user's
own (already hand-edited) GISIdeas_Asia_AlphaEarth_IMD.pptx, in place.

This does NOT regenerate the deck (build_gisideas_deck.py does that, from the
template, and would discard manual edits). It opens the existing file, adds
one slide built the same way as build_gisideas_deck.py's other image slides,
and inserts it right after the "Twenty rasters..." cross-city summary slide
if that title is found, else right before the "Thank you" slide, else at the
end. Every other slide is left untouched.

    C:\\ProgramData\\anaconda3\\python.exe add_all_predictions_slide.py
"""
import sys
sys.stdout.reconfigure(encoding="utf-8")

from PIL import Image
from pptx import Presentation
from pptx.util import Emu, Inches

OUT = (r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious"
       r"\Ground_truth_validation_S2\Presentation\GISIdeas_Asia_AlphaEarth_IMD.pptx")
IMG = "figs_deck/all_predictions_grid.png"
TITLE = "What all twenty predictions look like"
AFTER_TITLE_CONTAINS = "twenty rasters"
BEFORE_TITLE_EQUALS = "thank you"

prs = Presentation(OUT)


def layout_named(name):
    for master in prs.slide_masters:
        for lay in master.slide_layouts:
            if lay.name == name:
                return lay
    raise KeyError(name)


L_TEXT = layout_named("Pagina base_ testo A")


def slide_title(slide):
    for ph in slide.placeholders:
        if ph.placeholder_format.idx == 0:
            return ph.text_frame.text.strip()
    return ""


def add_picture_fit(slide, path, x, y, w, h):
    iw, ih = Image.open(path).size
    ar = iw / ih
    box_ar = w / h
    if ar > box_ar:
        pw, ph = w, Emu(int(w / ar))
    else:
        ph, pw = h, Emu(int(h * ar))
    px = x + Emu(int((w - pw) / 2))
    py = y + Emu(int((h - ph) / 2))
    slide.shapes.add_picture(path, px, py, width=pw, height=ph)


# ---- build the new slide (appends at the end for now; reordered below) ----
s = prs.slides.add_slide(L_TEXT)
s.placeholders[0].text = TITLE
for idx, val in ((10, "GISIdeas Asia  \u00b7  Hanoi, Vietnam"),
                  (11, "Mapping IMD with AlphaEarth Embeddings")):
    if idx in [p.placeholder_format.idx for p in s.placeholders]:
        s.placeholders[idx].text = val
body = s.placeholders[13]
body._element.getparent().remove(body._element)
add_picture_fit(s, IMG, Inches(0.37), Inches(2.02), Inches(12.60), Inches(5.15))

# ---- reorder: move the new slide right after AFTER_TITLE_CONTAINS --------
xml_slides = prs.slides._sldIdLst
new_id = list(xml_slides)[-1]          # the one we just appended
xml_slides.remove(new_id)

titles = [(i, slide_title(sl)) for i, sl in enumerate(prs.slides)]
target_i = None
for i, t in titles:
    if AFTER_TITLE_CONTAINS in t.lower():
        target_i = i
        break
if target_i is None:
    for i, t in titles:
        if t.lower() == BEFORE_TITLE_EQUALS:
            target_i = i - 1
            break

all_ids = list(xml_slides)
if target_i is None:
    all_ids.append(new_id)
    print("Anchor slide not found; appended at the very end.")
else:
    all_ids.insert(target_i + 1, new_id)
    print(f"Inserted after slide {target_i + 1} ({titles[target_i][1]!r}).")

for sid in list(xml_slides):
    xml_slides.remove(sid)
for sid in all_ids:
    xml_slides.append(sid)

prs.save(OUT)
print("saved:", OUT)
print("slides now:", len(prs.slides))

## 3. Add the Milan GHS-BUILT-S same-source convergence table (in place)

In [ ]:
# -*- coding: utf-8 -*-
"""Add the Milan GHS-BUILT-S same-source convergence table to the (already
hand-edited) GISIdeas_Asia_AlphaEarth_IMD.pptx, in place. Inserted right
after "Does the training label itself matter? Milan says yes" and before
"Twenty rasters, three cities, one reference" -- same table layout style
("Pagina base_Oggetto") used for the CLMS Milan table earlier in the deck.
Every other slide is left untouched.

    C:\\ProgramData\\anaconda3\\python.exe add_ghsl_convergence_slide_gisideas.py
"""
import sys
sys.stdout.reconfigure(encoding="utf-8")

from pptx import Presentation
from pptx.util import Inches

OUT = (r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious"
       r"\Ground_truth_validation_S2\Presentation\GISIdeas_Asia_AlphaEarth_IMD.pptx")

AFTER_TITLE = "does the training label itself matter? milan says yes"
DATE_TXT = "GISIdeas Asia  \u00b7  Hanoi, Vietnam"
FOOTER_TXT = "Mapping IMD with AlphaEarth Embeddings"

ROWS = [  # name, RMSE, MAE, R2, Bias
    ("S2 percentile (50 bands)",         "18.21", "13.52", "0.74", "+0.34"),
    ("S2 stack (40 bands)",              "18.29", "13.79", "0.73", "\u22120.14"),
    ("AlphaEarth embeddings (64 bands)", "18.36", "13.82", "0.73", "+0.34"),
    ("S2 median (10 bands)",             "18.79", "14.13", "0.72", "+0.39"),
]

prs = Presentation(OUT)


def layout_named(name):
    for master in prs.slide_masters:
        for lay in master.slide_layouts:
            if lay.name == name:
                return lay
    raise KeyError(name)


L_TABLE = layout_named("Pagina base_Oggetto")


def slide_title(slide):
    for ph in slide.placeholders:
        if ph.placeholder_format.idx == 0:
            return ph.text_frame.text.strip()
    return ""


def add_table(slide, idx, headers, rows):
    ph = slide.placeholders[idx]
    x, y, w, h = ph.left, ph.top, ph.width, ph.height
    ph._element.getparent().remove(ph._element)
    gframe = slide.shapes.add_table(len(rows) + 1, len(headers), x, y, w, h)
    table = gframe.table
    for j, hd in enumerate(headers):
        table.cell(0, j).text = str(hd)
    for i, row in enumerate(rows):
        for j, v in enumerate(row):
            table.cell(i + 1, j).text = str(v)
    return table


# ---- build the new slide (appended, then reordered) -----------------------
s = prs.slides.add_slide(L_TABLE)
s.placeholders[0].text = "Against GHS-BUILT-S, the four feature sets nearly converge"
for idx, val in ((10, DATE_TXT), (11, FOOTER_TXT)):
    if idx in [p.placeholder_format.idx for p in s.placeholders]:
        s.placeholders[idx].text = val
add_table(s, 1, ["Predictor set", "RMSE", "MAE", "R\u00b2", "Bias"], ROWS)
s.placeholders[16].text = ("Note: random forest, 998-point spatial holdout, scored against "
    "GHS-BUILT-S on an independently resampled point set. Spread: 0.58 RMSE (3.1% of the "
    "worst map), against 4.66 RMSE (33.0%) on the CLMS-trained table above \u2014 GHS-BUILT-S "
    "is the noisier label, so which features you feed it barely matters.")

# ---- reorder: move it right after AFTER_TITLE ------------------------------
xml_slides = prs.slides._sldIdLst
all_ids = list(xml_slides)
new_id = all_ids[-1]
all_ids = all_ids[:-1]

titles = [slide_title(sl).strip().lower() for sl in list(prs.slides)[:-1]]
anchor_i = titles.index(AFTER_TITLE)
new_order = all_ids[:anchor_i + 1] + [new_id] + all_ids[anchor_i + 1:]

for sid in list(xml_slides):
    xml_slides.remove(sid)
for sid in new_order:
    xml_slides.append(sid)

prs.save(OUT)
print("saved:", OUT)

prs2 = Presentation(OUT)
print("total slides:", len(prs2.slides))
for i, sl in enumerate(prs2.slides, 1):
    print(i, "|", slide_title(sl)[:70])

## 4. Fix the title-slide author line to match the professor's exact wording (in place)

In [ ]:
# -*- coding: utf-8 -*-
"""Fix the title-slide author line to match Prof. Brovelli's exact wording
(her email spells it "Xiao Tao", not "Xiao Tan" as used elsewhere in this
project -- kept as given rather than silently corrected). Patches the live
file in place; does not regenerate the deck.

    C:\ProgramData\anaconda3\python.exe add_authorline_patch.py
"""
from pptx import Presentation

OUT = (r"C:\Users\user\OneDrive - Politecnico di Milano\00LCZ\Matej_impervious"
       r"\Ground_truth_validation_S2\Presentation\GISIdeas_Asia_AlphaEarth_IMD.pptx")

prs = Presentation(OUT)
s = prs.slides[0]
changed = False
for ph in s.placeholders:
    tf = ph.text_frame
    if "Xiao Tan " in tf.text:
        for para in tf.paragraphs:
            joined = "".join(r.text for r in para.runs)
            if "Xiao Tan " in joined:
                new_text = joined.replace("Xiao Tan ", "Xiao Tao ")
                runs = list(para.runs)
                f = runs[0].font
                size, bold, italic, name = f.size, f.bold, f.italic, f.name
                try:
                    rgb = f.color.rgb
                except Exception:
                    rgb = None
                for r in runs:
                    r._r.getparent().remove(r._r)
                r = para.add_run()
                r.text = new_text
                if size is not None:
                    r.font.size = size
                r.font.bold = bold
                r.font.italic = italic
                if name:
                    r.font.name = name
                if rgb is not None:
                    r.font.color.rgb = rgb
                changed = True

if not changed:
    raise SystemExit("'Xiao Tan ' not found on the title slide -- wording may already differ.")

prs.save(OUT)
print("patched author line, saved:", OUT)